<a href="https://www.kaggle.com/code/dulapurkaystha/snack-guardian-ai?scriptVersionId=280164990" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# Snack Guardian AI: A Multi-Agent Gut-Friendly Snack Assistant

In [1]:
import os
from kaggle_secrets import UserSecretsClient

try:
    GOOGLE_API_KEY = UserSecretsClient().get_secret("GOOGLE_API_KEY")
    os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
    print("✅ Gemini API key setup complete.")
except Exception as e:
    print(
        f"🔑 Authentication Error: Please make sure you have added 'GOOGLE_API_KEY' to your Kaggle secrets. Details: {e}"
    )

✅ Gemini API key setup complete.


In [2]:
from google.adk.agents import Agent, SequentialAgent
from google.adk.models.google_llm import Gemini
from google.adk.runners import InMemoryRunner
from google.adk.tools import AgentTool, FunctionTool, google_search
from google.genai import types

print("✅ ADK components imported successfully.")

retry_config=types.HttpRetryOptions(
    attempts=5,  # Maximum retry attempts
    exp_base=7,  # Delay multiplier
    initial_delay=1,
    http_status_codes=[429, 500, 503, 504], # Retry on these HTTP errors
)

model = Gemini(
    model_name="gemini-2.5-flash-lite",
    retry_options=retry_config
)

✅ ADK components imported successfully.


In [3]:
import json
from typing import Dict, Any

# Very simple in-memory "database"
user_profiles: Dict[str, Dict[str, Any]] = {
    "default_user": {
        "name": ["Dula"],
        "gut_conditions": ["GERD"],
        "diet_preferences": ["vegetarian", "no onion", "no garlic"],
        "known_triggers": ["tomato", "citrus", "paprika"],
        "safe_foods": ["peanut butter", "banana", "rice"],
        "likes": ["warm", "gentle"],
        "dislikes": ["mint"],
        "never_suggest": [],
        "notes": "Starter profile for testing."
    }
}
print("✅ Profile created")

✅ Profile created


In [4]:
def get_user_profile(user_id:str) -> Dict[str, Any]:
    """
    Return the stored gut and snack profile for a given user.
    
    Args:
        user_id: A string ID for the user (e.g. "default_user")
    
    Returns:
        A dictionary containing gut conditions, triggers, safe foods, likes, dislikes, etc.
        If the user profile is not found, returns an empty dictionary.
    
    """
    return user_profiles.get(user_id,{})

#debug
print("Test: get_user_profile('default_user'):\n")
print(get_user_profile("default_user"))

Test: get_user_profile('default_user'):

{'name': ['Dula'], 'gut_conditions': ['GERD'], 'diet_preferences': ['vegetarian', 'no onion', 'no garlic'], 'known_triggers': ['tomato', 'citrus', 'paprika'], 'safe_foods': ['peanut butter', 'banana', 'rice'], 'likes': ['warm', 'gentle'], 'dislikes': ['mint'], 'never_suggest': [], 'notes': 'Starter profile for testing.'}


In [5]:
root_agent = Agent(
    name="helpful_snack_agent",
    model=model,
    # description="A simple snack agent that can suggest snacks."
    instruction="""
        You are a helpful assistant.
        When asked about the user's gut profile or preferences, 
        call get_user_profile(user_id="default_user") BEFORE answering.
    """,
    tools=[get_user_profile],
)

print("✅ Root Agent defined.")

✅ Root Agent defined.


In [6]:
runner = InMemoryRunner(agent=root_agent)

print("✅ Runner created.")

✅ Runner created.


In [7]:
# response = await runner.run_debug(
#     "I have mild acid reflux. Can you suggest a gentle evening snack?"
# )

response = await runner.run_debug(
    "What gut triggers do you currently know about me?"
)




 ### Created new session: debug_session_id

User > What gut triggers do you currently know about me?


helpful_snack_agent > I know that your gut triggers include tomato, citrus, and paprika.
